# Training auf Google Colab

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO = "https://github.com/wieland-github/information_extraction_german_medical_data.git"
!rm -rf /content/information_extraction_german_medical_data
!git clone {REPO} /content/information_extraction_german_medical_data
%cd /content/information_extraction_german_medical_data

In [ ]:
!pip install -q "spacy[cuda12x,transformers]>=3.8.14,<3.9.0" "datasets>=2.16.0,<3.0.0"

In [ ]:
!python related_work/scripts/data_preparation.py --outdir related_work/data
!ls -lh related_work/data

In [ ]:
MODEL = "deepset/gbert-large" 
TEST  = False                   # True = 50 steps, False = volles Training

name  = MODEL.split("/")[-1] + ("-test" if TEST else "")
OUT   = f"./models/{name}"
extra = "--training.max_steps 50 --training.eval_frequency 25 --training.patience 0" if TEST else ""

%cd /content/information_extraction_german_medical_data/related_work
cmd = (
    f"python -m spacy train config_spacy.cfg "
    f"--components.transformer.model.name {MODEL} "
    f"--output {OUT} "
    f"--paths.train ./data/train.spacy "
    f"--paths.dev ./data/validation.spacy "
    f"--gpu-id 0 "
    f"--training.optimizer.learn_rate.initial_rate 5e-5 "
    f"--training.batcher.size 128 {extra}"
)
print(cmd)
!{cmd}

In [ ]:
import shutil, os

src     = f"/content/information_extraction_german_medical_data/related_work/models/{name}/model-best"
dst_dir = "/content/drive/MyDrive/gbert_colab"
os.makedirs(dst_dir, exist_ok=True)
dst = f"{dst_dir}/{name}-model-best"

shutil.make_archive(dst, "zip", src)
print("Gespeichert in Drive:", dst + ".zip")